# Yellowhammer Reaction--Diffusion PINN --- Field Reconstruction

Reproducible Colab notebook for the field-reconstruction experiment of the
paper *A Distributed Parameter Identification Approach for Habitat-Modulated
Population Dynamics via Physics-Informed Neural Networks*.

**Usage.** Upload the three data files (`u_GLM_Bruant_jaune.mat`,
`Map_France.mat`, `Yellowhammer_2002_2024.csv`) to the Colab session, then
run all cells. The notebook writes its Python modules to disk, trains the
PINN, prints the metrics and recovered coefficients, and generates the paper
figures.

**Requirements.** A GPU runtime is recommended (`Runtime > Change runtime
type > GPU`).


## 1 · Environment


In [ ]:
import os, sys, time, json, glob, zipfile, numpy as np, torch
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import pandas as pd
print('torch', torch.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(os.cpu_count() or 1)
os.makedirs('out', exist_ok=True)



## 2 · Locate the data files

This cell looks for the three data files anywhere under `/content` (including inside any uploaded
`.zip`, which it will extract automatically) and in your Google Drive if mounted. If they are not
found it opens an upload dialog — select the three files (or a zip containing them).


In [ ]:
# Locate data files (works in Colab, Jupyter, or plain local run)
import os, sys
NEEDED = ["u_GLM_Bruant_jaune.mat", "Map_France.mat", "Yellowhammer_2002_2024.csv"]

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    if in_colab():
        from google.colab import files
        print("Please upload:", missing)
        files.upload()
    else:
        sys.exit("Missing data files: %s\n"
                 "Download them from the Yellowhammer benchmark:\n"
                 "  https://github.com/lias-laboratory/yellowhammer-benchmark\n"
                 "and place them next to this notebook." % missing)
assert all(os.path.exists(f) for f in NEEDED), "Data files still missing."
print("All data files present:", NEEDED)


## 3 · Write the modules to disk

These two `%%writefile` cells create `habitat_field.py` and `pinn_yellowhammer.py` in the session,
which makes `from pinn_yellowhammer import ...` work without any external package.


In [ ]:
%%writefile habitat_field.py
"""
habitat_field.py
================
Continuous, differentiable query of the habitat-quality index H(t, x, y) and its
spatial gradient (dH/dx, dH/dy) for arbitrary scattered points.

The construction reproduces *exactly* the methodology in
`Construction_of_the_habitat_index.ipynb`:

    H0   = u_GLM / u_sat                       (in [0,1])
    Z    = logit(clip(H0, eps, 1-eps))
    Z_es = (G_sigma_s * (Z * M)) / (G_sigma_s * M)        # mask-normalized spatial smoothing
    Z~(t)= sum_k w_k(t) Z_es[:,:,k] / sum_k w_k(t)        # temporal Gaussian weighting
    H    = sigmoid(Z~)
    grad H = H(1-H) grad Z~                                # chain rule

The only addition is a fast scattered-point evaluation:
  * the 17 spatial fields Z_es, dZ_es/dx, dZ_es/dy are precomputed on the 2 km grid
    (exactly as in the notebook), NaNs are nearest-filled so bilinear interpolation
    stays clean just inside the boundary,
  * for a query (year, x, y) we bilinearly interpolate the 17 slices in space and
    apply the *exact* temporal Gaussian weights -> this matches H_at_year /
    spatial_derivatives_H_at_year to machine precision at grid nodes.
"""

import numpy as np
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter, distance_transform_edt
from scipy.interpolate import RegularGridInterpolator
from matplotlib.path import Path


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def _split_nan_segments(x, y):
    isn = np.isnan(x) | np.isnan(y)
    idx = np.where(isn)[0]
    segments, start = [], 0
    for j in idx:
        if j - start >= 3:
            segments.append((x[start:j], y[start:j]))
        start = j + 1
    if len(x) - start >= 3:
        segments.append((x[start:], y[start:]))
    return segments


def _nearest_fill(arr2d, valid):
    """Fill ~valid cells with value of nearest valid cell (so interp doesn't see NaN)."""
    if valid.all():
        return arr2d
    _, (ii, jj) = distance_transform_edt(~valid, return_indices=True)
    return arr2d[ii, jj]


class HabitatField:
    def __init__(self, u_mat_path, map_mat_path,
                 epsilon=1e-6, sigma_s=4.0, sigma_t=1.0,
                 u_sat=15.0, year0=2002, dx_km=2.0, eta_B=1e-12):
        u_data = loadmat(u_mat_path)
        m_data = loadmat(map_mat_path)

        self.u_GLM = np.asarray(u_data["u_GLM"], dtype=np.float64)      # (ny, nx, nk)
        Mat_lat = np.asarray(u_data["Mat_lat"], dtype=np.float64)       # (ny, nx)
        Mat_long = np.asarray(u_data["Mat_long"], dtype=np.float64)
        Xfr = np.asarray(m_data["Xfr"], dtype=np.float64).squeeze()
        Yfr = np.asarray(m_data["Yfr"], dtype=np.float64).squeeze()

        self.eps, self.sigma_s, self.sigma_t = epsilon, sigma_s, sigma_t
        self.u_sat, self.year0 = float(u_sat), int(year0)
        ny, nx, nk = self.u_GLM.shape
        self.ny, self.nx, self.nk = ny, nx, nk
        self.tk = np.arange(nk, dtype=np.float64)

        # Regular 1D axes (grid is a regular 2 km mesh with NaN padding off-domain)
        self.lat_ax = np.nanmean(Mat_lat, axis=1)   # length ny, increasing
        self.lon_ax = np.nanmean(Mat_long, axis=1)  # placeholder; fix below
        self.lon_ax = np.nanmean(Mat_long, axis=0)  # length nx, increasing
        self.x_min, self.x_max = self.lon_ax.min(), self.lon_ax.max()
        self.y_min, self.y_max = self.lat_ax.min(), self.lat_ax.max()

        # France mask Omega on the grid
        self.M = self._mask_from_contour(Mat_long, Mat_lat, Xfr, Yfr)
        self.Xfr, self.Yfr = Xfr, Yfr
        Mf = self.M.astype(np.float64)

        dx = dy = float(dx_km)  # km units for derivative scaling (as in the notebook)

        # logit field, zeroed outside Omega for convolution
        H0 = self.u_GLM / self.u_sat
        H_eps = np.clip(H0, self.eps, 1.0 - self.eps)
        Z = np.log(H_eps / (1.0 - H_eps)) * Mf[:, :, None]

        B = np.maximum(gaussian_filter(Mf, sigma_s, mode="constant", cval=0.0), eta_B)
        B_y = gaussian_filter(Mf, sigma_s, order=(1, 0), mode="constant", cval=0.0) / dy
        B_x = gaussian_filter(Mf, sigma_s, order=(0, 1), mode="constant", cval=0.0) / dx

        Zes = np.empty((ny, nx, nk))
        dZes_dx = np.empty((ny, nx, nk))
        dZes_dy = np.empty((ny, nx, nk))
        for k in range(nk):
            Ak = gaussian_filter(Z[:, :, k], sigma_s, mode="constant", cval=0.0)
            A_y = gaussian_filter(Z[:, :, k], sigma_s, order=(1, 0), mode="constant", cval=0.0) / dy
            A_x = gaussian_filter(Z[:, :, k], sigma_s, order=(0, 1), mode="constant", cval=0.0) / dx
            Zes[:, :, k] = Ak / B
            # gradient of a quotient
            dZes_dx[:, :, k] = (A_x * B - Ak * B_x) / (B ** 2)
            dZes_dy[:, :, k] = (A_y * B - Ak * B_y) / (B ** 2)

        # derivatives above are per-km (dx_km). Convert to per-metre for PDE in metres.
        dZes_dx /= 1000.0
        dZes_dy /= 1000.0

        # nearest-fill outside the mask so bilinear interp near the coast is clean
        valid = self.M
        for k in range(nk):
            Zes[:, :, k] = _nearest_fill(Zes[:, :, k], valid)
            dZes_dx[:, :, k] = _nearest_fill(dZes_dx[:, :, k], valid)
            dZes_dy[:, :, k] = _nearest_fill(dZes_dy[:, :, k], valid)

        self.Zes, self.dZes_dx, self.dZes_dy = Zes, dZes_dx, dZes_dy

        # bilinear interpolators on (lat, lon) -> per-slice
        pts = (self.lat_ax, self.lon_ax)
        self._iZ = [RegularGridInterpolator(pts, Zes[:, :, k], bounds_error=False,
                    fill_value=None) for k in range(nk)]
        self._iZx = [RegularGridInterpolator(pts, dZes_dx[:, :, k], bounds_error=False,
                     fill_value=None) for k in range(nk)]
        self._iZy = [RegularGridInterpolator(pts, dZes_dy[:, :, k], bounds_error=False,
                     fill_value=None) for k in range(nk)]

        # initial-condition density field g = GLM density at slice 0 (~year0), filled+interp
        g0 = _nearest_fill(self.u_GLM[:, :, 0], valid)
        self._ig0 = RegularGridInterpolator(pts, g0, bounds_error=False, fill_value=None)

        # mask interpolator (for inside-France rejection sampling)
        self._imask = RegularGridInterpolator(pts, Mf, bounds_error=False, fill_value=0.0)

    # ----------------------------------------------------------------- mask
    @staticmethod
    def _mask_from_contour(Mat_long, Mat_lat, Xfr, Yfr):
        base = np.isfinite(Mat_long) & np.isfinite(Mat_lat)
        xmin, xmax = np.nanmin(Xfr), np.nanmax(Xfr)
        ymin, ymax = np.nanmin(Yfr), np.nanmax(Yfr)
        bbox = base & (Mat_long >= xmin) & (Mat_long <= xmax) & \
               (Mat_lat >= ymin) & (Mat_lat <= ymax)
        pts = np.column_stack([Mat_long[bbox].ravel(), Mat_lat[bbox].ravel()])
        inside = np.zeros(pts.shape[0], dtype=bool)
        for xs, ys in _split_nan_segments(Xfr, Yfr):
            poly = np.column_stack([xs, ys])
            if poly.shape[0] < 3:
                continue
            if not (poly[0, 0] == poly[-1, 0] and poly[0, 1] == poly[-1, 1]):
                poly = np.vstack([poly, poly[0]])
            inside |= Path(poly).contains_points(pts)
        M = np.zeros(Mat_long.shape, dtype=bool)
        M[bbox] = inside
        return M

    def _temporal_weights(self, year):
        year = np.atleast_1d(np.asarray(year, dtype=np.float64))
        t_index = year - self.year0                      # (Q,)
        tau = t_index[:, None] - self.tk[None, :]        # (Q, nk)
        w = np.exp(-0.5 * (tau / self.sigma_t) ** 2)
        return w / np.maximum(w.sum(axis=1, keepdims=True), 1e-300)

    # ------------------------------------------------------------ scattered query
    def query(self, year, x, y):
        """Return H, dH/dx, dH/dy (per metre) at scattered points (year, x, y)."""
        x = np.asarray(x, dtype=np.float64).ravel()
        y = np.asarray(y, dtype=np.float64).ravel()
        year = np.asarray(year, dtype=np.float64).ravel()
        P = np.column_stack([y, x])                      # interpolators expect (lat, lon)
        nk = self.nk
        Zek = np.stack([self._iZ[k](P) for k in range(nk)], axis=1)   # (Q, nk)
        Zxk = np.stack([self._iZx[k](P) for k in range(nk)], axis=1)
        Zyk = np.stack([self._iZy[k](P) for k in range(nk)], axis=1)
        w = self._temporal_weights(year)                              # (Q, nk)
        Ze = (w * Zek).sum(axis=1)
        dZe_dx = (w * Zxk).sum(axis=1)
        dZe_dy = (w * Zyk).sum(axis=1)
        H = _sigmoid(Ze)
        fac = H * (1.0 - H)
        return H, fac * dZe_dx, fac * dZe_dy

    def g_initial(self, x, y):
        """Initial density field g(x,y) (GLM density at year0) at scattered points."""
        x = np.asarray(x, dtype=np.float64).ravel()
        y = np.asarray(y, dtype=np.float64).ravel()
        return self._ig0(np.column_stack([y, x]))

    def inside(self, x, y, thresh=0.5):
        x = np.asarray(x, dtype=np.float64).ravel()
        y = np.asarray(y, dtype=np.float64).ravel()
        return self._imask(np.column_stack([y, x])) >= thresh

    def H_at_year_grid(self, year):
        """Full-grid H (for plotting / FEM-style comparison). NaN outside France."""
        w = self._temporal_weights(year)[0]
        Ze = np.tensordot(self.Zes, w, axes=(2, 0))
        H = _sigmoid(Ze)
        H[~self.M] = np.nan
        return H

    def sample_glm(self, n, seed=0):
        """Sample the dense GLM density field as (year, x, y, u) inside France.
        Slice k is mapped to calendar year year0 + k (notebook convention)."""
        rng = np.random.default_rng(seed)
        ii, jj = np.where(self.M)
        nk = self.nk
        ki = rng.integers(0, nk, size=n)
        si = rng.integers(0, len(ii), size=n)
        I, J = ii[si], jj[si]
        year = self.year0 + ki.astype(np.float64)
        x = self.lon_ax[J]; y = self.lat_ax[I]
        u = self.u_GLM[I, J, ki]
        return year, x, y, u

    def glm_grid_at_slice(self, k):
        """Return (X, Y, U) for GLM slice k (NaN outside France) for plotting/metrics."""
        U = self.u_GLM[:, :, k].copy()
        U[~self.M] = np.nan
        X, Y = np.meshgrid(self.lon_ax, self.lat_ax)
        return X, Y, U



In [ ]:
%%writefile pinn_yellowhammer.py
"""
pinn_yellowhammer.py
====================
Physics-Informed Neural Network for the Yellowhammer (Bruant jaune) population
density field u(t,x,y) over mainland France (2002-2024), with simultaneous
inverse identification of the ecological parameters theta = (D0, alpha, rmax,
beta, Kmax, gamma).

Governing PDE (reaction-diffusion with habitat-dependent coefficients):

    du/dt = div( a(H) grad u ) + r(H) u (1 - u/K(H))            (+ optional advection)
      a(H) = D0 (1 - alpha H)
      r(H) = rmax (beta + (1-beta) H)
      K(H) = Kmax (gamma + (1-gamma) H)

KEY ENGINEERING CHOICES (these are what make it converge to good results):

 1. Full non-dimensionalization. Coordinates are in Lambert metres (~1e6) and
    years (~1e1); raw, the PDE is catastrophically ill-conditioned for a NN.
    We map x,y -> [0,1] with a single length scale L (isotropy preserved) and
    t -> [0,1], and density u -> n = u/U. The network learns DIMENSIONLESS
    parameters; physical values are recovered analytically afterwards.

 2. Positivity & sign-constrained parameters. The output is softplus(.) so
    n >= 0. Parameters are reparameterized so that a(H)>0, r(H)>0, K(H)>0 hold
    for ALL H in [0,1] by construction (alpha,beta,gamma in (0,1); D0,rmax,Kmax>0).
    Guaranteeing a(H)>0 removes the backward-diffusion ill-posedness that is the
    usual cause of advection/diffusion instability in this class of problem.

 3. Habitat enters only through precomputed coefficients (H, gradH at the fixed
    collocation points), NOT as a network input, so the autodiff residual is exact.

 4. Adam (warm, with residual-weight ramp) followed by L-BFGS polishing.
"""

import numpy as np
import torch
import torch.nn as nn
from dataclasses import dataclass, field
from scipy.stats import qmc

from habitat_field import HabitatField


# --------------------------------------------------------------------------- config
@dataclass
class Config:
    u_mat: str = "u_GLM_Bruant_jaune.mat"
    map_mat: str = "Map_France.mat"
    csv: str = "Yellowhammer_2002_2024.csv"

    year0: int = 2002
    year1: int = 2024
    U: float = 15.0                     # density scale (u_sat)

    # network
    layers: tuple = (3, 64, 64, 64, 64, 1)
    fourier_m: int = 0                  # >0 enables Fourier features of size m on (x,y)
    fourier_sigma: float = 3.0

    # sampling
    n_collocation: int = 20000
    n_ic: int = 4000
    max_data: int = 0                   # 0 = use all observations
    val_frac: float = 0.15              # held-out fraction of observations for validation
    use_glm_data: bool = True           # fit dense denoised GLM field (identifiable inverse)
    n_glm: int = 15000                  # total GLM field samples; split train/test by train_frac
    train_frac: float = 0.70            # disjoint 70/30 split of the GLM field (train/test)
    fix_Kmax_tilde: float = 1.2         # if >0, Kmax is fixed (dimensionless); else learned
    fix_D0_tilde: float = -1.0          # if >=0, D0_tilde is fixed (for profile likelihood)
    n_boundary: int = 2000
    use_bc: bool = True
    use_advection: bool = False         # taxis term -div(chi*gradH * u); off => matches PDF

    # loss weights
    w_res: float = 1.0
    w_data: float = 1.0
    w_ic: float = 1.0
    w_bc: float = 0.1
    res_ramp_steps: int = 1500          # ramp residual weight 0->w_res over these steps
    # weak ecological prior on dimensionless diffusion D0_tilde (Tikhonov anchor).
    # disabled when w_prior<=0 or D0_tilde_prior<0. Calibration:
    # D0_phys[km2/yr] = D0_tilde * (L/1000)^2 / T.
    w_prior: float = 0.0
    D0_tilde_prior: float = -1.0

    # optimization
    adam_steps: int = 8000
    adam_lr: float = 2e-3
    lbfgs_steps: int = 1500
    seed: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# --------------------------------------------------------------------------- model
class MLP(nn.Module):
    def __init__(self, layers, fourier_m=0, fourier_sigma=3.0):
        super().__init__()
        self.fourier_m = fourier_m
        in_dim = layers[0]
        if fourier_m > 0:
            # random Fourier features on the 2 spatial coords (cols 1,2)
            B = torch.randn(2, fourier_m) * fourier_sigma
            self.register_buffer("Bff", B)
            in_dim = layers[0] + 2 * fourier_m
        seq, dims = [], [in_dim] + list(layers[1:])
        for i in range(len(dims) - 2):
            seq += [nn.Linear(dims[i], dims[i + 1]), nn.Tanh()]
        seq += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, txy):
        if self.fourier_m > 0:
            xy = txy[:, 1:3]
            proj = 2 * np.pi * xy @ self.Bff
            feats = torch.cat([txy, torch.sin(proj), torch.cos(proj)], dim=1)
        else:
            feats = txy
        return torch.nn.functional.softplus(self.net(feats))   # density >= 0


class PINN(nn.Module):
    """Holds the field network and the trainable dimensionless parameters."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.net = MLP(cfg.layers, cfg.fourier_m, cfg.fourier_sigma)
        # unconstrained raw parameters -> constrained via softplus / sigmoid
        self.pD = nn.Parameter(torch.tensor(-2.0))   # D0_tilde  = softplus(pD)   ~0.13
        self._fixD = cfg.fix_D0_tilde
        if self._fixD >= 0:
            self.register_buffer("Dfix", torch.tensor(float(self._fixD)))
        self.pa = nn.Parameter(torch.tensor(0.0))     # alpha     = sigmoid(pa)    =0.5
        self.pr = nn.Parameter(torch.tensor(0.0))     # rmax_tilde= softplus(pr)   ~0.69
        self.pb = nn.Parameter(torch.tensor(0.0))     # beta      = sigmoid(pb)
        self._fixK = cfg.fix_Kmax_tilde
        if self._fixK > 0:
            self.register_buffer("Kfix", torch.tensor(float(self._fixK)))
        else:
            self.pK = nn.Parameter(torch.tensor(0.0))  # Kmax_tilde= softplus(pK)
        self.pg = nn.Parameter(torch.tensor(0.0))     # gamma     = sigmoid(pg)
        if cfg.use_advection:
            self.pchi = nn.Parameter(torch.tensor(-2.0))  # chi_tilde = softplus

    # constrained, dimensionless parameters
    def params(self):
        sp = torch.nn.functional.softplus
        sg = torch.sigmoid
        Kmax = self.Kfix if self._fixK > 0 else (sp(self.pK) + 0.05)
        D0 = self.Dfix if self._fixD >= 0 else sp(self.pD)
        p = dict(D0=D0, alpha=sg(self.pa), rmax=sp(self.pr),
                 beta=sg(self.pb), Kmax=Kmax, gamma=sg(self.pg))
        if self.cfg.use_advection:
            p["chi"] = sp(self.pchi)
        return p


# --------------------------------------------------------------------------- driver
class Trainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
        self.dev = torch.device(cfg.device)

        self.hf = HabitatField(cfg.u_mat, cfg.map_mat, year0=cfg.year0)
        # normalization constants
        self.x0, self.y0 = self.hf.x_min, self.hf.y_min
        self.L = max(self.hf.x_max - self.hf.x_min, self.hf.y_max - self.hf.y_min)
        self.T = float(cfg.year1 - cfg.year0)
        self.U = cfg.U

        self.model = PINN(cfg).to(self.dev)
        self._build_points()

    # ---- normalization helpers
    def to_norm(self, year, x, y):
        return ((year - self.cfg.year0) / self.T,
                (x - self.x0) / self.L,
                (y - self.y0) / self.L)

    def _tensor(self, a):
        return torch.tensor(np.asarray(a, np.float64), dtype=torch.float32, device=self.dev)

    # ---- point sets
    def _lhs_inside(self, n, t_lo, t_hi):
        """Latin-Hypercube sample (year,x,y) rejected to inside France."""
        out_y, out_x, out_t = [], [], []
        eng = qmc.LatinHypercube(d=3, seed=self.cfg.seed)
        need = n
        while need > 0:
            s = eng.random(max(need * 3, 1000))
            yr = self.cfg.year0 + s[:, 0] * self.T
            xx = self.hf.x_min + s[:, 1] * (self.hf.x_max - self.hf.x_min)
            yy = self.hf.y_min + s[:, 2] * (self.hf.y_max - self.hf.y_min)
            if t_lo == t_hi:
                yr[:] = t_lo
            m = self.hf.inside(xx, yy)
            out_t.append(yr[m]); out_x.append(xx[m]); out_y.append(yy[m])
            need = n - sum(len(a) for a in out_t)
        t = np.concatenate(out_t)[:n]; x = np.concatenate(out_x)[:n]; y = np.concatenate(out_y)[:n]
        return t, x, y

    def _build_points(self):
        c = self.cfg
        # ----- collocation
        t, x, y = self._lhs_inside(c.n_collocation, c.year0, c.year1)
        H, Hx, Hy = self.hf.query(t, x, y)
        tn, xn, yn = self.to_norm(t, x, y)
        self.col = dict(
            txy=self._tensor(np.column_stack([tn, xn, yn])),
            H=self._tensor(H), Hx=self._tensor(Hx * self.L), Hy=self._tensor(Hy * self.L))
        self.col["txy"].requires_grad_(True)

        # ----- data: dense denoised GLM field (identifiable inverse), or sparse counts
        import pandas as pd
        df = pd.read_csv(c.csv)
        if c.use_glm_data:
            # Draw the GLM field points ONCE, then split disjointly into
            # train_frac (train) and 1-train_frac (held-out test). This
            # guarantees the test points never appear in training.
            yr, xx, yy, uu = self.hf.sample_glm(c.n_glm, seed=c.seed)
            rng = np.random.default_rng(c.seed)
            perm = rng.permutation(len(uu))
            n_tr = int(c.train_frac * len(uu))
            tr, te = perm[:n_tr], perm[n_tr:]

            # --- training set (train_frac of the field) ---
            tn, xn, yn = self.to_norm(yr[tr], xx[tr], yy[tr])
            self.data = dict(
                txy=self._tensor(np.column_stack([tn, xn, yn])),
                n=self._tensor(uu[tr] / self.U).reshape(-1, 1))

            # --- disjoint held-out test set (1-train_frac of the field) ---
            tn, xn, yn = self.to_norm(yr[te], xx[te], yy[te])
            self.val_glm = dict(
                txy=self._tensor(np.column_stack([tn, xn, yn])), u=uu[te])

            # ALL counts held out for independent validation (never seen in training)
            dv = df
            tn, xn, yn = self.to_norm(dv.Year.values, dv.Longitude.values, dv.Latitude.values)
            self.val = dict(
                txy=self._tensor(np.column_stack([tn, xn, yn])),
                n=self._tensor(dv.EMBCIT.values / self.U).reshape(-1, 1),
                counts=dv.EMBCIT.values.astype(float))
        else:
            if c.max_data and len(df) > c.max_data:
                df = df.sample(c.max_data, random_state=c.seed)
            df = df.sample(frac=1.0, random_state=c.seed).reset_index(drop=True)
            n_val = int(c.val_frac * len(df))
            df_val, df_tr = df.iloc[:n_val], df.iloc[n_val:]
            tn, xn, yn = self.to_norm(df_tr.Year.values, df_tr.Longitude.values, df_tr.Latitude.values)
            self.data = dict(
                txy=self._tensor(np.column_stack([tn, xn, yn])),
                n=self._tensor(df_tr.EMBCIT.values / self.U).reshape(-1, 1))
            tn, xn, yn = self.to_norm(df_val.Year.values, df_val.Longitude.values, df_val.Latitude.values)
            self.val = dict(
                txy=self._tensor(np.column_stack([tn, xn, yn])),
                n=self._tensor(df_val.EMBCIT.values / self.U).reshape(-1, 1),
                counts=df_val.EMBCIT.values.astype(float))

        # ----- initial condition (t = year0)
        t, x, y = self._lhs_inside(c.n_ic, c.year0, c.year0)
        g = self.hf.g_initial(x, y)
        tn, xn, yn = self.to_norm(t, x, y)
        self.ic = dict(
            txy=self._tensor(np.column_stack([tn, xn, yn])),
            n=self._tensor(g / self.U).reshape(-1, 1))

        # ----- boundary (Neumann) points + inward normals
        if c.use_bc:
            self._build_boundary()

    def _build_boundary(self):
        Xfr, Yfr = self.hf.Xfr, self.hf.Yfr
        finite = np.isfinite(Xfr) & np.isfinite(Yfr)
        xs, ys = Xfr[finite], Yfr[finite]
        # tangents -> normals (rotate 90 deg), point a touch inside
        tx = np.gradient(xs); ty = np.gradient(ys)
        nrm = np.hypot(tx, ty) + 1e-9
        nx, ny = ty / nrm, -tx / nrm                # one normal direction
        # ensure inward: nudge and test mask
        eps = 1500.0
        test_in = self.hf.inside(xs + eps * nx, ys + eps * ny)
        nx = np.where(test_in, nx, -nx); ny = np.where(test_in, ny, -ny)
        idx = np.random.choice(len(xs), min(self.cfg.n_boundary, len(xs)), replace=False)
        bx, by = xs[idx] + eps * nx[idx], ys[idx] + eps * ny[idx]   # just inside
        bnx, bny = nx[idx], ny[idx]
        keep = self.hf.inside(bx, by)
        bx, by, bnx, bny = bx[keep], by[keep], bnx[keep], bny[keep]
        H, Hx, Hy = self.hf.query(np.full(len(bx), self.cfg.year0 + self.T / 2), bx, by)
        tn = np.full(len(bx), 0.5)
        xn, yn = (bx - self.x0) / self.L, (by - self.y0) / self.L
        self.bc = dict(
            txy=self._tensor(np.column_stack([tn, xn, yn])),
            nrm=self._tensor(np.column_stack([bnx, bny])))   # (gx,gy) normal in physical -> same dir in norm
        self.bc["txy"].requires_grad_(True)

    # ---- derivatives & residual
    @staticmethod
    def _grads(n, txy):
        g = torch.autograd.grad(n, txy, torch.ones_like(n), create_graph=True)[0]
        return g[:, 0:1], g[:, 1:2], g[:, 2:3]    # n_t, n_x, n_y  (normalized coords)

    def residual(self):
        p = self.model.params()
        txy = self.col["txy"]
        n = self.model.net(txy)
        n_t, n_x, n_y = self._grads(n, txy)
        n_xx = torch.autograd.grad(n_x, txy, torch.ones_like(n_x), create_graph=True)[0][:, 1:2]
        n_yy = torch.autograd.grad(n_y, txy, torch.ones_like(n_y), create_graph=True)[0][:, 2:3]
        H = self.col["H"].reshape(-1, 1)
        Hx = self.col["Hx"].reshape(-1, 1); Hy = self.col["Hy"].reshape(-1, 1)

        a = p["D0"] * (1.0 - p["alpha"] * H)          # a(H) > 0 by construction
        a_prime = -p["D0"] * p["alpha"]
        div_term = a * (n_xx + n_yy) + a_prime * (Hx * n_x + Hy * n_y)

        r = p["rmax"] * (p["beta"] + (1.0 - p["beta"]) * H)
        K = p["Kmax"] * (p["gamma"] + (1.0 - p["gamma"]) * H)
        reaction = r * n * (1.0 - n / K)

        res = n_t - div_term - reaction
        if self.cfg.use_advection:
            # taxis flux J = -chi * gradH * n  ->  -div(J) added to du/dt
            # div(chi gradH n) = chi (Hxx+Hyy) n + chi (Hx n_x + Hy n_y); we approximate
            # the habitat Laplacian as negligible at NN scale and keep advective transport:
            chi = p["chi"]
            res = res + chi * (Hx * n_x + Hy * n_y)
        return res

    def losses(self, step=None):
        c = self.cfg
        res = self.residual()
        L_res = (res ** 2).mean()
        nd = self.model.net(self.data["txy"])
        L_data = ((nd - self.data["n"]) ** 2).mean()
        ni = self.model.net(self.ic["txy"])
        L_ic = ((ni - self.ic["n"]) ** 2).mean()
        if c.use_bc:
            txy = self.bc["txy"]; nb = self.model.net(txy)
            g = torch.autograd.grad(nb, txy, torch.ones_like(nb), create_graph=True)[0]
            dn_dn = g[:, 1:2] * self.bc["nrm"][:, 0:1] + g[:, 2:3] * self.bc["nrm"][:, 1:2]
            L_bc = (dn_dn ** 2).mean()
        else:
            L_bc = torch.tensor(0.0, device=self.dev)
        # residual-weight ramp (lets data/IC organize the field first)
        wr = c.w_res
        if step is not None and c.res_ramp_steps > 0:
            wr = c.w_res * min(1.0, step / c.res_ramp_steps)
        total = wr * L_res + c.w_data * L_data + c.w_ic * L_ic + c.w_bc * L_bc
        L_prior = torch.tensor(0.0, device=self.dev)
        if c.w_prior > 0 and c.D0_tilde_prior >= 0:
            D0t = self.model.params()["D0"]
            L_prior = (D0t - c.D0_tilde_prior) ** 2
            total = total + c.w_prior * L_prior
        return total, dict(res=L_res.item(), data=L_data.item(),
                           ic=L_ic.item(), bc=float(L_bc.detach()),
                           prior=float(L_prior.detach()) if torch.is_tensor(L_prior) else float(L_prior),
                           total=float(total.detach()))

    # ---- training
    def train(self, verbose=True):
        c = self.cfg
        opt = torch.optim.Adam(self.model.parameters(), lr=c.adam_lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, c.adam_steps, eta_min=c.adam_lr * 0.02)
        hist = []
        self.param_hist = []
        for step in range(c.adam_steps):
            opt.zero_grad()
            total, log = self.losses(step)
            total.backward()
            opt.step(); sched.step()
            if verbose and (step % max(1, c.adam_steps // 20) == 0 or step == c.adam_steps - 1):
                p = {k: float(v) for k, v in self.model.params().items()}
                print(f"[adam {step:5d}] tot={log['total']:.3e} res={log['res']:.2e} "
                      f"data={log['data']:.2e} ic={log['ic']:.2e} bc={log['bc']:.2e} "
                      f"| D0~={p['D0']:.3f} a={p['alpha']:.2f} r~={p['rmax']:.3f} "
                      f"b={p['beta']:.2f} K~={p['Kmax']:.2f} g={p['gamma']:.2f}")
            hist.append(log)
            if step % 25 == 0:
                self.param_hist.append((step, {k: float(v) for k, v in self.model.params().items()}))

        if c.lbfgs_steps > 0:
            opt2 = torch.optim.LBFGS(self.model.parameters(), max_iter=c.lbfgs_steps,
                                     history_size=50, line_search_fn="strong_wolfe",
                                     tolerance_grad=1e-9, tolerance_change=1e-12)
            def closure():
                opt2.zero_grad()
                total, _ = self.losses(step=c.res_ramp_steps)
                total.backward()
                return total
            opt2.step(closure)
            _, log = self.losses(step=c.res_ramp_steps)
            if verbose:
                p = {k: float(v) for k, v in self.model.params().items()}
                print(f"[lbfgs done] tot={log['total']:.3e} res={log['res']:.2e} "
                      f"data={log['data']:.2e} ic={log['ic']:.2e} | "
                      f"D0~={p['D0']:.3f} r~={p['rmax']:.3f} K~={p['Kmax']:.2f}")
            hist.append(log)
        self.hist = hist
        return hist

    # ---- validation metrics (held-out observations, in count units)
    @torch.no_grad()
    def val_metrics(self):
        pred = self.model.net(self.val["txy"]).cpu().numpy().ravel() * self.U
        obs = self.val["counts"]
        rmse = float(np.sqrt(np.mean((pred - obs) ** 2)))
        mae = float(np.mean(np.abs(pred - obs)))
        ss_res = np.sum((obs - pred) ** 2)
        ss_tot = np.sum((obs - obs.mean()) ** 2)
        r2 = float(1 - ss_res / ss_tot)
        corr = float(np.corrcoef(pred, obs)[0, 1])
        return dict(rmse=rmse, mae=mae, r2=r2, corr=corr, pred=pred, obs=obs)

    @torch.no_grad()
    def val_glm_metrics(self):
        if not hasattr(self, "val_glm"):
            return None
        pred = self.model.net(self.val_glm["txy"]).cpu().numpy().ravel() * self.U
        u = self.val_glm["u"]
        rmse = float(np.sqrt(np.mean((pred - u) ** 2)))
        ss_res = np.sum((u - pred) ** 2); ss_tot = np.sum((u - u.mean()) ** 2)
        return dict(rmse=rmse, r2=float(1 - ss_res / ss_tot),
                    corr=float(np.corrcoef(pred, u)[0, 1]))

    # ---- physical parameter recovery
    def physical_params(self):
        p = {k: float(v) for k, v in self.model.params().items()}
        L_km = self.L / 1000.0
        return dict(
            D0_km2_per_yr=p["D0"] * L_km ** 2 / self.T,
            alpha=p["alpha"],
            rmax_per_yr=p["rmax"] / self.T,
            beta=p["beta"],
            Kmax_birds=p["Kmax"] * self.U,
            gamma=p["gamma"],
            **({"chi_km2_per_yr": p["chi"] * L_km ** 2 / self.T} if "chi" in p else {}))

    # ---- prediction on a grid (for plotting / metrics)
    @torch.no_grad()
    def predict_grid(self, year, nx=160, ny=170):
        xs = np.linspace(self.hf.x_min, self.hf.x_max, nx)
        ys = np.linspace(self.hf.y_min, self.hf.y_max, ny)
        X, Y = np.meshgrid(xs, ys)
        inside = self.hf.inside(X.ravel(), Y.ravel())
        tn, xn, yn = self.to_norm(np.full(X.size, year), X.ravel(), Y.ravel())
        txy = self._tensor(np.column_stack([tn, xn, yn]))
        n = self.model.net(txy).cpu().numpy().ravel() * self.U
        n[~inside] = np.nan
        return X, Y, n.reshape(Y.shape)


## 4 · Train (final “Run D” configuration)

Field-data term on the denoised GLM field, Fourier-feature network, light physics weight
`λ_f = 0.15`. The raw counts are held out entirely for independent validation.


In [ ]:
from pinn_yellowhammer import Config, Trainer

# ============================================================
#  BEST-OF-N MODEL SELECTION (levers 1+2+3 combined)
#  1. multi-seed loop        -> robust to D0 wandering / local minima
#  2. more field points      -> n_glm larger, still a disjoint 70/30 split
#  3. longer optimisation    -> more Adam + more L-BFGS polishing
#  The run with the best held-out (disjoint 70/30) field R2 is kept as `tr`.
# ============================================================

SEEDS = [0, 1, 2, 3, 4]        # lever 1

def make_cfg(seed):
    return Config(
        n_collocation=5000, n_ic=2500, n_boundary=1200,
        use_glm_data=True, n_glm=60000, train_frac=0.70,   # lever 2 (60k -> 42k train / 18k test)
        fix_Kmax_tilde=1.2,
        layers=(3, 128, 128, 128, 128, 1), fourier_m=96, fourier_sigma=5.0,
        w_res=0.15, w_data=6.0, w_ic=3.0, w_bc=0.05, res_ramp_steps=900,
        adam_steps=5000, adam_lr=3e-3, lbfgs_steps=500,    # lever 3 (longer)
        seed=seed,
    )

best = None            # (r2, trainer, seed, metrics)
results = []
t0 = time.time()
for s in SEEDS:
    cfg_s = make_cfg(s)
    tr_s = Trainer(cfg_s)
    tr_s.train(verbose=False)                 # quiet inside the loop
    vg = tr_s.val_glm_metrics()               # disjoint 70/30 test field
    results.append((s, vg['r2'], vg['corr'], vg['rmse']))
    print('seed %d  ->  held-out field  R2=%.4f  corr=%.4f  RMSE=%.4f'
          % (s, vg['r2'], vg['corr'], vg['rmse']))
    if best is None or vg['r2'] > best[0]:
        best = (vg['r2'], tr_s, s, vg)

# expose the best model as `tr` so all downstream cells work unchanged
tr  = best[1]
cfg = tr.cfg
print('\n' + '='*56)
print('BEST seed = %d   held-out field R2 = %.4f' % (best[2], best[0]))
print('GLM field: %d train / %d test (70/30 disjoint)'
      % (int(cfg.train_frac*cfg.n_glm), cfg.n_glm - int(cfg.train_frac*cfg.n_glm)))
print('independent held-out raw counts: %d' % len(tr.val['counts']))
print('total time: %.1f min' % ((time.time()-t0)/60))
print('='*56)


## 5 · Metrics and recovered parameters


In [ ]:
vm = tr.val_metrics(); vg = tr.val_glm_metrics(); phys = tr.physical_params()
print('held-out GLM field : RMSE=%.3f  R2=%.3f  corr=%.3f' % (vg['rmse'], vg['r2'], vg['corr']))
print('held-out raw counts: RMSE=%.3f  MAE=%.3f  R2=%.3f  corr=%.3f' % (vm['rmse'], vm['mae'], vm['r2'], vm['corr']))
print('\nrecovered physical parameters:')
for k, v in phys.items(): print('   %-16s = %.4g' % (k, float(v)))

# numpy-safe JSON (val dicts hold np.float64 / ndarray)
def _py(o):
    import numpy as _np
    if isinstance(o, dict):  return {k: _py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_py(v) for v in o]
    if isinstance(o, _np.ndarray):   return o.tolist()
    if isinstance(o, _np.generic):   return o.item()
    return o

# strip the large per-point arrays (pred/obs) so summary.json stays small;
# scalar metrics are kept. The arrays remain in `vm` for the figures.
vm_scalar = {k: v for k, v in vm.items() if k not in ('pred', 'obs')}
json.dump(_py(dict(val_glm=vg, val_counts=vm_scalar, physical_params=phys)),
          open('out/summary.json', 'w'), indent=2)
torch.save(tr.model.state_dict(), 'out/pinn_weights.pt')
print('saved out/summary.json and out/pinn_weights.pt')


## 6 · Paper figures (1–4, 6)

Training diagnostics, multi-epoch density maps, PINN-vs-GLM calibration, national trajectory,
and the identified habitat-response curves.


In [ ]:
# ---- FIG 1: losses + parameter identification
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
H = tr.hist[:cfg.adam_steps]; it = np.arange(len(H))
for key, lab in [('total','total'),('res','PDE residual'),('data','data (GLM)'),('ic','initial cond.'),('bc','boundary')]:
    ax[0].semilogy(it, [h[key] for h in H], label=lab, lw=1.3)
ax[0].set_xlabel('Adam iteration'); ax[0].set_ylabel('loss'); ax[0].set_title('(a) Training losses')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
steps = [s for s,_ in tr.param_hist]
for key in ['D0','alpha','rmax','beta','gamma']:
    ax[1].plot(steps, [p[key] for _,p in tr.param_hist], label=key, lw=1.4)
ax[1].set_xlabel('Adam iteration'); ax[1].set_ylabel('dimensionless value')
ax[1].set_title('(b) Inverse parameter identification'); ax[1].legend(fontsize=8, ncol=2); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('out/fig1_convergence.pdf', dpi=140); plt.show()



In [ ]:
# ---- FIG 2: predicted density maps + raw observations, three epochs
df = pd.read_csv(cfg.csv); years = [2005, 2014, 2024]
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2)); grids = []; vmax = 0
for yr in years:
    X, Y, Z = tr.predict_grid(yr, nx=150, ny=160); grids.append((X,Y,Z)); vmax = max(vmax, np.nanpercentile(Z,99))
for axk, yr, (X,Y,Z) in zip(axes, years, grids):
    pc = axk.pcolormesh(X/1e3, Y/1e3, Z, cmap='viridis', norm=Normalize(0,vmax), shading='auto')
    axk.plot(tr.hf.Xfr/1e3, tr.hf.Yfr/1e3, 'k', lw=0.6)
    d = df[df.Year == yr]
    axk.scatter(d.Longitude/1e3, d.Latitude/1e3, c=d.EMBCIT, cmap='viridis', norm=Normalize(0,vmax), s=9, edgecolor='w', linewidth=0.2)
    axk.set_title(f'PINN density u(t,x,y), {yr}'); axk.set_aspect('equal'); axk.set_xlabel('x [km]'); axk.set_ylabel('y [km]')
    plt.colorbar(pc, ax=axk, fraction=0.046, label='birds / site')
plt.tight_layout(); plt.savefig('out/fig2_density_maps.pdf', dpi=140); plt.show()



In [ ]:
# ---- FIG 3: PINN vs GLM reference (2014) + held-out calibration
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
Xg, Yg, Ug = tr.hf.glm_grid_at_slice(12)
_, _, Zp = tr.predict_grid(2014, nx=Ug.shape[1], ny=Ug.shape[0]); vmx = np.nanpercentile(Ug,99)
for a, F, ttl in [(ax[0], Ug, 'GLM reference, 2014'), (ax[1], Zp, 'PINN reconstruction, 2014')]:
    pc = a.pcolormesh(Xg/1e3, Yg/1e3, F, cmap='viridis', norm=Normalize(0,vmx), shading='auto')
    a.plot(tr.hf.Xfr/1e3, tr.hf.Yfr/1e3, 'k', lw=0.6); a.set_aspect('equal'); a.set_title(ttl)
    a.set_xlabel('x [km]'); a.set_ylabel('y [km]'); plt.colorbar(pc, ax=a, fraction=0.046)
pr = tr.model.net(tr.val_glm['txy']).detach().cpu().numpy().ravel() * tr.U
ax[2].scatter(tr.val_glm['u'], pr, s=6, alpha=0.2, color='#2c7fb8')
m = max(tr.val_glm['u'].max(), pr.max()); ax[2].plot([0,m],[0,m],'r--',lw=1)
ax[2].set_xlabel('GLM density (held-out)'); ax[2].set_ylabel('PINN predicted')
ax[2].set_title(f"Calibration  R2={vg['r2']:.2f}"); ax[2].grid(alpha=0.3); ax[2].set_aspect('equal')
plt.tight_layout(); plt.savefig('out/fig3_pinn_vs_glm.pdf', dpi=140); plt.show()



In [ ]:
# ---- FIG 4: national mean-density trajectory
yrs = np.arange(2002, 2025); pinn_mean = [np.nanmean(tr.predict_grid(int(y), nx=120, ny=130)[2]) for y in yrs]
glm_mean = [np.nanmean(tr.hf.glm_grid_at_slice(k)[2]) for k in range(tr.hf.nk)]
cnt_mean = pd.read_csv(cfg.csv).groupby('Year').EMBCIT.mean()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(yrs, pinn_mean, '-o', ms=3, label='PINN mean density', color='#d95f02')
ax.plot(np.arange(2002, 2002+tr.hf.nk), glm_mean, '-s', ms=3, label='GLM mean density', color='#1b9e77')
ax.plot(cnt_mean.index, cnt_mean.values, ':^', ms=3, label='mean raw count', color='#7570b3', alpha=0.8)
ax.axvspan(2018.5, 2024.5, color='gray', alpha=0.12, label='extrapolation (no GLM)')
ax.set_xlabel('year'); ax.set_ylabel('mean density (birds / site)'); ax.set_title('National-scale Yellowhammer trajectory')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('out/fig4_trajectory.pdf', dpi=140); plt.show()



In [ ]:
# ---- FIG 6: identified habitat-response curves
p = {k: float(v) for k, v in tr.model.params().items()}; L_km = tr.L/1000.0; Hh = np.linspace(0,1,200)
a_phys = p['D0']*(1-p['alpha']*Hh)*L_km**2/tr.T
r_phys = (p['rmax']*(p['beta']+(1-p['beta'])*Hh))/tr.T
K_phys = (p['Kmax']*(p['gamma']+(1-p['gamma'])*Hh))*tr.U
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].plot(Hh, r_phys, lw=2, color='#1b9e77'); ax[0].set_title('(a) Growth rate r(H)'); ax[0].set_xlabel('habitat index H'); ax[0].set_ylabel('r [1/yr]'); ax[0].grid(alpha=0.3)
ax[1].plot(Hh, K_phys, lw=2, color='#d95f02'); ax[1].set_title('(b) Carrying capacity K(H)'); ax[1].set_xlabel('habitat index H'); ax[1].set_ylabel('K [birds/site]'); ax[1].grid(alpha=0.3)
ax[2].plot(Hh, a_phys, lw=2, color='#7570b3'); ax[2].set_title('(c) Diffusivity a(H) (weakly id.)'); ax[2].set_xlabel('habitat index H'); ax[2].set_ylabel('a [km^2/yr]'); ax[2].grid(alpha=0.3)
print('r: %.4f -> %.4f /yr (x%.1f) | K: %.2f -> %.2f (x%.1f)' % (r_phys[0],r_phys[-1],r_phys[-1]/r_phys[0],K_phys[0],K_phys[-1],K_phys[-1]/K_phys[0]))
plt.tight_layout(); plt.savefig('out/fig6_habitat_response.pdf', dpi=140); plt.show()



## 7 · Extra professional figures

Additional visualizations requested: predictions across **many years**, fidelity per year,
the habitat driver field, and a fine multi-year comparison against the GLM reference.


In [ ]:
# ---- EXTRA 1: full multi-year prediction panel (every year 2002-2024, shared colorbar)
years_all = list(range(2002, 2025))
grids = [tr.predict_grid(y, nx=110, ny=120) for y in years_all]
vmax = np.nanpercentile(np.concatenate([g[2][~np.isnan(g[2])] for g in grids]), 99)
ncol = 6; nrow = int(np.ceil(len(years_all)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.5*ncol, 2.7*nrow))
for k, (yr, (X,Y,Z)) in enumerate(zip(years_all, grids)):
    a = axes.ravel()[k]
    pc = a.pcolormesh(X/1e3, Y/1e3, Z, cmap='viridis', norm=Normalize(0,vmax), shading='auto')
    a.plot(tr.hf.Xfr/1e3, tr.hf.Yfr/1e3, 'k', lw=0.4)
    a.set_title(str(yr), fontsize=9); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
    if yr > 2018: a.spines[:].set_color('crimson'); a.spines[:].set_linewidth(1.6)
for k in range(len(years_all), nrow*ncol): axes.ravel()[k].axis('off')

cax = fig.add_axes([1.01, 0.15, 0.015, 0.7]); plt.colorbar(pc, cax=cax, label='birds / site')
plt.tight_layout(); plt.savefig('out/figE1_all_years.pdf', dpi=140, bbox_inches='tight'); plt.show()



In [ ]:
# ---- EXTRA 2: per-year reconstruction fidelity vs the GLM field
def metrics(a, b):
    m = ~(np.isnan(a) | np.isnan(b)); a, b = a[m], b[m]
    r2 = 1 - np.sum((a-b)**2)/np.sum((a-a.mean())**2); rmse = np.sqrt(np.mean((a-b)**2))
    return r2, rmse, np.corrcoef(a,b)[0,1]
rows = []
for k in range(tr.hf.nk):
    Xg, Yg, Ug = tr.hf.glm_grid_at_slice(k)
    _, _, Zp = tr.predict_grid(2002+k, nx=Ug.shape[1], ny=Ug.shape[0])
    r2, rmse, c = metrics(Ug, Zp); rows.append((2002+k, r2, rmse, c))
yr = [r[0] for r in rows]
fig, ax1 = plt.subplots(figsize=(9, 4.2))
ax1.bar(yr, [r[1] for r in rows], color='#2c7fb8', alpha=0.85, label='R^2')
ax1.set_ylabel('R^2', color='#2c7fb8'); ax1.set_ylim(0,0.8); ax1.set_xlabel('year')
ax2 = ax1.twinx(); ax2.plot(yr, [r[2] for r in rows], '-o', color='#d95f02', ms=4, label='RMSE')
ax2.set_ylabel('RMSE [birds]', color='#d95f02')
ax1.set_title('Per-year reconstruction fidelity (PINN vs denoised GLM field)')
plt.tight_layout(); plt.savefig('out/figE2_per_year.pdf', dpi=140); plt.show()



In [ ]:
# ---- EXTRA 3: habitat driver field H at four epochs
epochs = [2002, 2008, 2014, 2018]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for a, yr in zip(axes, epochs):
    Hg = tr.hf.H_at_year_grid(yr); X, Y = np.meshgrid(tr.hf.lon_ax, tr.hf.lat_ax)
    pc = a.pcolormesh(X/1e3, Y/1e3, Hg, cmap='YlGn', norm=Normalize(0,1), shading='auto')
    a.plot(tr.hf.Xfr/1e3, tr.hf.Yfr/1e3, 'k', lw=0.6); a.set_aspect('equal')
    a.set_title(f'Habitat index H, {yr}'); a.set_xlabel('x [km]'); a.set_ylabel('y [km]')
    plt.colorbar(pc, ax=a, fraction=0.046, label='H')
plt.tight_layout(); plt.savefig('out/figE3_habitat_index.pdf', dpi=140); plt.show()



In [ ]:
# ---- EXTRA 4: PINN vs GLM side-by-side for several years (rows = years)
yrs_cmp = [2002, 2008, 2014, 2018]
fig, axes = plt.subplots(len(yrs_cmp), 3, figsize=(13, 3.4*len(yrs_cmp)))
for i, yr in enumerate(yrs_cmp):
    k = yr - 2002; Xg, Yg, Ug = tr.hf.glm_grid_at_slice(k)
    _, _, Zp = tr.predict_grid(yr, nx=Ug.shape[1], ny=Ug.shape[0]); v = np.nanpercentile(Ug,99)
    diff = Zp - Ug
    for j,(F,ttl,cmap,nm) in enumerate([(Ug,'GLM',\
          'viridis',Normalize(0,v)),(Zp,'PINN','viridis',Normalize(0,v)),\
          (diff,'PINN - GLM','RdBu_r',Normalize(-v/2,v/2))]):
        a = axes[i,j]; pc = a.pcolormesh(Xg/1e3, Yg/1e3, F, cmap=cmap, norm=nm, shading='auto')
        a.plot(tr.hf.Xfr/1e3, tr.hf.Yfr/1e3, 'k', lw=0.5); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
        if i == 0: a.set_title(ttl)
        if j == 0: a.set_ylabel(str(yr), fontsize=12)
        plt.colorbar(pc, ax=a, fraction=0.046)
#fig.suptitle('PINN vs GLM reference and residual, by year', y=1.002, fontsize=13)
plt.tight_layout(); plt.savefig('out/figE4_pinn_vs_glm_years.pdf', dpi=140, bbox_inches='tight'); plt.show()



In [ ]:
# ---- EXTRA 4: PINN vs GLM side-by-side for several years (rows = years)

yrs_cmp = [2002, 2008, 2014, 2018]

fig, axes = plt.subplots(
    len(yrs_cmp),
    3,
    figsize=(10.5, 10.5)   # more compact for article
)

for i, yr in enumerate(yrs_cmp):

    k = yr - 2002
    Xg, Yg, Ug = tr.hf.glm_grid_at_slice(k)

    _, _, Zp = tr.predict_grid(
        yr,
        nx=Ug.shape[1],
        ny=Ug.shape[0]
    )

    v = np.nanpercentile(Ug, 99)
    diff = Zp - Ug

    fields = [
        (Ug,   'GLM',        'viridis', Normalize(0, v)),
        (Zp,   'PINN',       'viridis', Normalize(0, v)),
        (diff, 'PINN - GLM', 'RdBu_r',  Normalize(-v/2, v/2))
    ]

    for j, (F, ttl, cmap, nm) in enumerate(fields):

        a = axes[i, j]

        pc = a.pcolormesh(
            Xg / 1e3,
            Yg / 1e3,
            F,
            cmap=cmap,
            norm=nm,
            shading='auto',
            rasterized=True   # IMPORTANT: strongly reduces PDF size
        )

        a.plot(
            tr.hf.Xfr / 1e3,
            tr.hf.Yfr / 1e3,
            'k',
            lw=0.6
        )

        a.set_aspect('equal')
        a.set_xticks([])
        a.set_yticks([])

        if i == 0:
            a.set_title(ttl, fontsize=11)

        if j == 0:
            a.set_ylabel(str(yr), fontsize=11)

        cbar = plt.colorbar(
            pc,
            ax=a,
            fraction=0.046,
            pad=0.03
        )

        cbar.ax.tick_params(labelsize=8)

plt.tight_layout(
    rect=[0, 0, 1, 0.98]
)

# Save publication-quality hybrid PDF
fig.savefig(
    'out/figE4_pinn_vs_glm_years.pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.02
)

plt.show()

## 8 · Download all outputs

Zips everything in `./out/` (figures, `summary.json`, trained weights) for download.


In [ ]:
# Save outputs (download in Colab, keep on disk locally)
import os, shutil, zipfile
os.makedirs("outputs", exist_ok=True)
for f in os.listdir("."):
    if f.endswith(".png") or f.endswith(".pdf"):
        shutil.copy(f, os.path.join("outputs", f))
with zipfile.ZipFile("yellowhammer_outputs.zip", "w") as z:
    for f in os.listdir("outputs"):
        z.write(os.path.join("outputs", f))
print("Figures saved in ./outputs/ and zipped to yellowhammer_outputs.zip")

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False
if in_colab():
    from google.colab import files
    files.download("yellowhammer_outputs.zip")


In [ ]:
import zipfile, os
with zipfile.ZipFile('yellowhammer_pinn_outputs.zip') as z:
    print(z.namelist())   # voir ce qu'il y a dedans
    # si pinn_weights.pt est dans la liste, l'extraire :
    if 'pinn_weights.pt' in z.namelist():
        z.extract('pinn_weights.pt')
        print('pinn_weights.pt extrait !')

In [ ]:
# ============================================================
#  TABLEAU DE VALIDATION COMPLET (avec MAE) + PARAMETRES
#  A lancer soit apres l'entrainement (tr existe deja),
#  soit en chargeant les poids sauvegardes pinn_weights.pt
# ============================================================
import torch, numpy as np
from pinn_yellowhammer import Config, Trainer

# --- Choix : utiliser tr existant, ou recharger depuis les poids ---
USE_SAVED_WEIGHTS = True   # True = charger pinn_weights.pt (verif exacte du seed 4)

if USE_SAVED_WEIGHTS:
    cfg = Config(layers=(3,128,128,128,128,1), fourier_m=96, fourier_sigma=5.0,
                 n_glm=60000, train_frac=0.70, fix_Kmax_tilde=1.2, use_bc=True,
                 seed=4, adam_steps=0, lbfgs_steps=0)   # 0 iter : on ne fait que charger
    tr = Trainer(cfg)
    tr.model.load_state_dict(torch.load('pinn_weights.pt', map_location=tr.dev))
tr.model.eval()

# --- CHAMP debruite held-out (test disjoint 30%) ---
vg = tr.val_glm_metrics()
with torch.no_grad():
    pred_f = tr.model.net(tr.val_glm['txy']).cpu().numpy().ravel() * tr.U
u_f = tr.val_glm['u']
mae_field = float(np.mean(np.abs(pred_f - u_f)))

# --- COMPTAGES bruts held-out ---
vm = tr.val_metrics()

# --- Affichage type "Table de validation" ---
print("="*64)
print(f"{'Target':<30}{'RMSE':>8}{'MAE':>8}{'R2':>9}{'corr':>8}")
print("-"*64)
print(f"{'Denoised field (disjoint test)':<30}{vg['rmse']:>8.2f}{mae_field:>8.2f}{vg['r2']:>9.3f}{vg['corr']:>8.3f}")
print(f"{'Raw counts (independent)':<30}{vm['rmse']:>8.2f}{vm['mae']:>8.2f}{vm['r2']:>9.3f}{vm['corr']:>8.3f}")
print("="*64)

# --- Parametres (pour tab:params_disjoint) ---
print("\nPARAMETRES PHYSIQUES:")
for k,v in tr.physical_params().items():
    print(f"  {k:<18} = {float(v):.4f}")
print("\nPARAMETRES DIMENSIONLESS:")
for k,v in tr.model.params().items():
    print(f"  {k:<8} = {float(v):.4f}")

In [ ]:
# ============================================================
# ALL FIGURES - PUBLICATION QUALITY + REDUCED PDF SIZE
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

os.makedirs("out", exist_ok=True)

# ------------------------------------------------------------
# Common PDF saving function
# ------------------------------------------------------------
def save_fig(fig, filename, dpi=300):
    fig.savefig(
        filename,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.02
    )


# ============================================================
# FIG 1: Losses + parameter identification
# Pure vector PDF
# ============================================================

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.8))

H = tr.hist[:cfg.adam_steps]
it = np.arange(len(H))

for key, lab in [
    ('total', 'total'),
    ('res', 'PDE residual'),
    ('data', 'data (GLM)'),
    ('ic', 'initial cond.'),
    ('bc', 'boundary')
]:
    ax[0].semilogy(
        it,
        [h[key] for h in H],
        label=lab,
        lw=1.3
    )

ax[0].set_xlabel('Adam iteration')
ax[0].set_ylabel('loss')
ax[0].set_title('(a) Training losses')
ax[0].legend(fontsize=8)
ax[0].grid(alpha=0.3)

steps = [s for s, _ in tr.param_hist]

for key in ['D0', 'alpha', 'rmax', 'beta', 'gamma']:
    ax[1].plot(
        steps,
        [p[key] for _, p in tr.param_hist],
        label=key,
        lw=1.4
    )

ax[1].set_xlabel('Adam iteration')
ax[1].set_ylabel('dimensionless value')
ax[1].set_title('(b) Inverse parameter identification')
ax[1].legend(fontsize=8, ncol=2)
ax[1].grid(alpha=0.3)

plt.tight_layout()

save_fig(
    fig,
    'out/fig1_convergence.pdf'
)

plt.show()



# ============================================================
# FIG 2: Predicted density maps + raw observations
# pcolormesh + scatter rasterized
# ============================================================

df = pd.read_csv(cfg.csv)
years = [2005, 2014, 2024]

fig, axes = plt.subplots(
    1, 3,
    figsize=(11.5, 4.2)
)

grids = []
vmax = 0

for yr in years:

    X, Y, Z = tr.predict_grid(
        yr,
        nx=150,
        ny=160
    )

    grids.append((X, Y, Z))

    vmax = max(
        vmax,
        np.nanpercentile(Z, 99)
    )


for axk, yr, (X, Y, Z) in zip(
    axes,
    years,
    grids
):

    pc = axk.pcolormesh(
        X / 1e3,
        Y / 1e3,
        Z,
        cmap='viridis',
        norm=Normalize(0, vmax),
        shading='auto',
        rasterized=True
    )

    axk.plot(
        tr.hf.Xfr / 1e3,
        tr.hf.Yfr / 1e3,
        'k',
        lw=0.6
    )

    d = df[df.Year == yr]

    axk.scatter(
        d.Longitude / 1e3,
        d.Latitude / 1e3,
        c=d.EMBCIT,
        cmap='viridis',
        norm=Normalize(0, vmax),
        s=9,
        edgecolor='w',
        linewidth=0.2,
        rasterized=True
    )

    axk.set_title(
        f'PINN density $u(t,x,y)$, {yr}'
    )

    axk.set_aspect('equal')

    axk.set_xlabel('x [km]')
    axk.set_ylabel('y [km]')

    plt.colorbar(
        pc,
        ax=axk,
        fraction=0.046,
        pad=0.03,
        label='birds / site'
    )

plt.tight_layout()

save_fig(
    fig,
    'out/fig2_density_maps.pdf'
)

plt.show()



# ============================================================
# FIG 3: PINN vs GLM reference + held-out calibration
# pcolormesh and scatter rasterized
# ============================================================

fig, ax = plt.subplots(
    1, 3,
    figsize=(11.5, 4.0)
)

Xg, Yg, Ug = tr.hf.glm_grid_at_slice(12)

_, _, Zp = tr.predict_grid(
    2014,
    nx=Ug.shape[1],
    ny=Ug.shape[0]
)

vmx = np.nanpercentile(
    Ug,
    99
)

for a, F, ttl in [
    (
        ax[0],
        Ug,
        'GLM reference, 2014'
    ),
    (
        ax[1],
        Zp,
        'PINN reconstruction, 2014'
    )
]:

    pc = a.pcolormesh(
        Xg / 1e3,
        Yg / 1e3,
        F,
        cmap='viridis',
        norm=Normalize(0, vmx),
        shading='auto',
        rasterized=True
    )

    a.plot(
        tr.hf.Xfr / 1e3,
        tr.hf.Yfr / 1e3,
        'k',
        lw=0.6
    )

    a.set_aspect('equal')
    a.set_title(ttl)

    a.set_xlabel('x [km]')
    a.set_ylabel('y [km]')

    plt.colorbar(
        pc,
        ax=a,
        fraction=0.046,
        pad=0.03
    )


pr = (
    tr.model
    .net(tr.val_glm['txy'])
    .detach()
    .cpu()
    .numpy()
    .ravel()
    * tr.U
)

ax[2].scatter(
    tr.val_glm['u'],
    pr,
    s=6,
    alpha=0.2,
    color='#2c7fb8',
    rasterized=True
)

m = max(
    tr.val_glm['u'].max(),
    pr.max()
)

ax[2].plot(
    [0, m],
    [0, m],
    'r--',
    lw=1
)

ax[2].set_xlabel(
    'GLM density (held-out)'
)

ax[2].set_ylabel(
    'PINN predicted'
)

ax[2].set_title(
    f"Calibration  $R^2$={vg['r2']:.2f}"
)

ax[2].grid(alpha=0.3)
ax[2].set_aspect('equal')

plt.tight_layout()

save_fig(
    fig,
    'out/fig3_pinn_vs_glm.pdf'
)

plt.show()



# ============================================================
# FIG 4: National mean-density trajectory
# Pure vector PDF
# ============================================================

yrs = np.arange(
    2002,
    2025
)

pinn_mean = [
    np.nanmean(
        tr.predict_grid(
            int(y),
            nx=120,
            ny=130
        )[2]
    )
    for y in yrs
]

glm_mean = [
    np.nanmean(
        tr.hf.glm_grid_at_slice(k)[2]
    )
    for k in range(tr.hf.nk)
]

cnt_mean = (
    pd.read_csv(cfg.csv)
    .groupby('Year')
    .EMBCIT
    .mean()
)

fig, ax = plt.subplots(
    figsize=(7.2, 4.0)
)

ax.plot(
    yrs,
    pinn_mean,
    '-o',
    ms=3,
    label='PINN mean density',
    color='#d95f02'
)

ax.plot(
    np.arange(
        2002,
        2002 + tr.hf.nk
    ),
    glm_mean,
    '-s',
    ms=3,
    label='GLM mean density',
    color='#1b9e77'
)

ax.plot(
    cnt_mean.index,
    cnt_mean.values,
    ':^',
    ms=3,
    label='mean raw count',
    color='#7570b3',
    alpha=0.8
)

ax.axvspan(
    2018.5,
    2024.5,
    color='gray',
    alpha=0.12,
    label='extrapolation (no GLM)'
)

ax.set_xlabel('year')

ax.set_ylabel(
    'mean density (birds / site)'
)

#ax.set_title(
    #'National-scale Yellowhammer trajectory'
#)

ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()

save_fig(
    fig,
    'out/fig4_trajectory.pdf'
)

plt.show()



# ============================================================
# FIG 6: Identified habitat-response curves
# Pure vector PDF
# ============================================================

p = {
    k: float(v)
    for k, v in tr.model.params().items()
}

L_km = tr.L / 1000.0

Hh = np.linspace(
    0,
    1,
    200
)

a_phys = (
    p['D0']
    * (1 - p['alpha'] * Hh)
    * L_km**2
    / tr.T
)

r_phys = (
    p['rmax']
    * (
        p['beta']
        + (1 - p['beta']) * Hh
    )
    / tr.T
)

K_phys = (
    p['Kmax']
    * (
        p['gamma']
        + (1 - p['gamma']) * Hh
    )
    * tr.U
)


fig, ax = plt.subplots(
    1, 3,
    figsize=(10.5, 3.7)
)

ax[0].plot(
    Hh,
    r_phys,
    lw=2,
    color='#1b9e77'
)

ax[0].set_title(
    '(a) Growth rate $r(H)$'
)

ax[0].set_xlabel(
    'habitat index $H$'
)

ax[0].set_ylabel(
    'r [1/yr]'
)

ax[0].grid(alpha=0.3)


ax[1].plot(
    Hh,
    K_phys,
    lw=2,
    color='#d95f02'
)

ax[1].set_title(
    '(b) Carrying capacity $K(H)$'
)

ax[1].set_xlabel(
    'habitat index $H$'
)

ax[1].set_ylabel(
    'K [birds/site]'
)

ax[1].grid(alpha=0.3)


ax[2].plot(
    Hh,
    a_phys,
    lw=2,
    color='#7570b3'
)

ax[2].set_title(
    '(c) Diffusivity $a(H)$'
)

ax[2].set_xlabel(
    'habitat index $H$'
)

ax[2].set_ylabel(
    'a [km$^2$/yr]'
)

ax[2].grid(alpha=0.3)


print(
    'r: %.4f -> %.4f /yr (x%.1f) | '
    'K: %.2f -> %.2f (x%.1f)'
    % (
        r_phys[0],
        r_phys[-1],
        r_phys[-1] / r_phys[0],
        K_phys[0],
        K_phys[-1],
        K_phys[-1] / K_phys[0]
    )
)

plt.tight_layout()

save_fig(
    fig,
    'out/fig6_habitat_response.pdf'
)

plt.show()



# ============================================================
# EXTRA 1: Full multi-year prediction panel
# VERY LARGE -> rasterized pcolormesh
# ============================================================

years_all = list(
    range(
        2002,
        2025
    )
)

grids = [
    tr.predict_grid(
        y,
        nx=110,
        ny=120
    )
    for y in years_all
]

vmax = np.nanpercentile(
    np.concatenate(
        [
            g[2][~np.isnan(g[2])]
            for g in grids
        ]
    ),
    99
)

ncol = 6

nrow = int(
    np.ceil(
        len(years_all) / ncol
    )
)

fig, axes = plt.subplots(
    nrow,
    ncol,
    figsize=(11.5, 7.8)
)

for k, (
    yr,
    (X, Y, Z)
) in enumerate(
    zip(
        years_all,
        grids
    )
):

    a = axes.ravel()[k]

    pc = a.pcolormesh(
        X / 1e3,
        Y / 1e3,
        Z,
        cmap='viridis',
        norm=Normalize(
            0,
            vmax
        ),
        shading='auto',
        rasterized=True
    )

    a.plot(
        tr.hf.Xfr / 1e3,
        tr.hf.Yfr / 1e3,
        'k',
        lw=0.4
    )

    a.set_title(
        str(yr),
        fontsize=9
    )

    a.set_aspect('equal')
    a.set_xticks([])
    a.set_yticks([])

    if yr > 2018:

        for spine in a.spines.values():

            spine.set_color(
                'crimson'
            )

            spine.set_linewidth(
                1.4
            )


for k in range(
    len(years_all),
    nrow * ncol
):

    axes.ravel()[k].axis(
        'off'
    )


fig.subplots_adjust(
    right=0.92,
    wspace=0.08,
    hspace=0.18
)

cax = fig.add_axes(
    [
        0.94,
        0.18,
        0.015,
        0.64
    ]
)

fig.colorbar(
    pc,
    cax=cax,
    label='birds / site'
)

save_fig(
    fig,
    'out/figE1_all_years.pdf'
)

plt.show()



# ============================================================
# EXTRA 2: Per-year reconstruction fidelity
# Pure vector PDF
# ============================================================

def metrics(a, b):

    m = ~(
        np.isnan(a)
        |
        np.isnan(b)
    )

    a = a[m]
    b = b[m]

    r2 = (
        1
        -
        np.sum(
            (a - b)**2
        )
        /
        np.sum(
            (a - a.mean())**2
        )
    )

    rmse = np.sqrt(
        np.mean(
            (a - b)**2
        )
    )

    corr = np.corrcoef(
        a,
        b
    )[0, 1]

    return (
        r2,
        rmse,
        corr
    )


rows = []

for k in range(
    tr.hf.nk
):

    Xg, Yg, Ug = (
        tr.hf.glm_grid_at_slice(k)
    )

    _, _, Zp = (
        tr.predict_grid(
            2002 + k,
            nx=Ug.shape[1],
            ny=Ug.shape[0]
        )
    )

    r2, rmse, c = metrics(
        Ug,
        Zp
    )

    rows.append(
        (
            2002 + k,
            r2,
            rmse,
            c
        )
    )


yr = [
    r[0]
    for r in rows
]

fig, ax1 = plt.subplots(
    figsize=(7.8, 3.8)
)

ax1.bar(
    yr,
    [r[1] for r in rows],
    color='#2c7fb8',
    alpha=0.85,
    label='$R^2$'
)

ax1.set_ylabel(
    '$R^2$',
    color='#2c7fb8'
)

ax1.set_ylim(
    0,
    0.8
)

ax1.set_xlabel(
    'year'
)

ax2 = ax1.twinx()

ax2.plot(
    yr,
    [r[2] for r in rows],
    '-o',
    color='#d95f02',
    ms=4,
    label='RMSE'
)

ax2.set_ylabel(
    'RMSE [birds]',
    color='#d95f02'
)

#ax1.set_title(
  #  'Per-year reconstruction fidelity '
   # '(PINN vs denoised GLM field)'
#)

plt.tight_layout()

save_fig(
    fig,
    'out/figE2_per_year.pdf'
)

plt.show()



# ============================================================
# EXTRA 3: Habitat driver H
# Large maps -> rasterized
# ============================================================

epochs = [
    2002,
    2008,
    2014,
    2018
]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(11.5, 3.6)
)

for a, yr in zip(
    axes,
    epochs
):

    Hg = tr.hf.H_at_year_grid(
        yr
    )

    X, Y = np.meshgrid(
        tr.hf.lon_ax,
        tr.hf.lat_ax
    )

    pc = a.pcolormesh(
        X / 1e3,
        Y / 1e3,
        Hg,
        cmap='YlGn',
        norm=Normalize(
            0,
            1
        ),
        shading='auto',
        rasterized=True
    )

    a.plot(
        tr.hf.Xfr / 1e3,
        tr.hf.Yfr / 1e3,
        'k',
        lw=0.6
    )

    a.set_aspect(
        'equal'
    )

    a.set_title(
        f'Habitat index $H$, {yr}',
        fontsize=10
    )

    a.set_xlabel(
        'x [km]'
    )

    a.set_ylabel(
        'y [km]'
    )

    plt.colorbar(
        pc,
        ax=a,
        fraction=0.046,
        pad=0.03,
        label='$H$'
    )

plt.tight_layout()

save_fig(
    fig,
    'out/figE3_habitat_index.pdf'
)

plt.show()



# ============================================================
# EXTRA 4: PINN vs GLM for several years
# Very large -> rasterized maps
# ============================================================

yrs_cmp = [
    2002,
    2008,
    2014,
    2018
]

fig, axes = plt.subplots(
    len(yrs_cmp),
    3,
    figsize=(9.5, 10.0)
)

for i, yr in enumerate(
    yrs_cmp
):

    k = yr - 2002

    Xg, Yg, Ug = (
        tr.hf.glm_grid_at_slice(k)
    )

    _, _, Zp = (
        tr.predict_grid(
            yr,
            nx=Ug.shape[1],
            ny=Ug.shape[0]
        )
    )

    v = np.nanpercentile(
        Ug,
        99
    )

    diff = Zp - Ug

    fields = [
        (
            Ug,
            'GLM',
            'viridis',
            Normalize(
                0,
                v
            )
        ),

        (
            Zp,
            'PINN',
            'viridis',
            Normalize(
                0,
                v
            )
        ),

        (
            diff,
            'PINN - GLM',
            'RdBu_r',
            Normalize(
                -v / 2,
                v / 2
            )
        )
    ]

    for j, (
        F,
        ttl,
        cmap,
        nm
    ) in enumerate(
        fields
    ):

        a = axes[i, j]

        pc = a.pcolormesh(
            Xg / 1e3,
            Yg / 1e3,
            F,
            cmap=cmap,
            norm=nm,
            shading='auto',
            rasterized=True
        )

        a.plot(
            tr.hf.Xfr / 1e3,
            tr.hf.Yfr / 1e3,
            'k',
            lw=0.5
        )

        a.set_aspect(
            'equal'
        )

        a.set_xticks([])
        a.set_yticks([])

        if i == 0:

            a.set_title(
                ttl,
                fontsize=11
            )

        if j == 0:

            a.set_ylabel(
                str(yr),
                fontsize=11
            )

        cbar = plt.colorbar(
            pc,
            ax=a,
            fraction=0.046,
            pad=0.025
        )

        cbar.ax.tick_params(
            labelsize=7
        )


#fig.suptitle(
  #  'PINN vs GLM reference and residual, by year',
   # y=0.995,
  #  fontsize=12
#)

plt.tight_layout(
    rect=[
        0,
        0,
        1,
        0.98
    ]
)

save_fig(
    fig,
    'out/figE4_pinn_vs_glm_years.pdf'
)

plt.show()


print("\n======================================")
print("All figures saved successfully in out/")
print("======================================")

In [ ]:
# Save outputs (download in Colab, keep on disk locally)
import os, shutil, zipfile
os.makedirs("outputs", exist_ok=True)
for f in os.listdir("."):
    if f.endswith(".png") or f.endswith(".pdf"):
        shutil.copy(f, os.path.join("outputs", f))
with zipfile.ZipFile("yellowhammer_outputs.zip", "w") as z:
    for f in os.listdir("outputs"):
        z.write(os.path.join("outputs", f))
print("Figures saved in ./outputs/ and zipped to yellowhammer_outputs.zip")

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False
if in_colab():
    from google.colab import files
    files.download("yellowhammer_outputs.zip")
